In [5]:
import torch
import torch.nn as nn
import snntorch as snn

device = torch.device("mps")

class SNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(10, 2)
        self.leaky = snn.Leaky(beta=0.9)

    def forward(self, x):
        mem = self.leaky.init_leaky()

        current = self.fc(x)
        spike, mem = self.leaky(current, mem)

        return spike, mem

model = SNN().to(device)

evenX = torch.randint(0, 10, (32, 10)) * 2
oddX = torch.randint(0, 10, (32, 10)) * 2 + 1

evenY = torch.ones(32, dtype=torch.long)
oddY = torch.zeros(32, dtype=torch.long)

x = torch.cat((evenX, oddX), dim=0).float().to(device)
y = torch.cat ((evenY, oddY), dim=0).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters())

model.train()

for epoch in range(100):
    optimizer.zero_grad()

    spike, mem = model(x)
    loss = criterion(mem, y)

    loss.backward()
    optimizer.step()

    prediction = mem.argmax(dim=1)
    accuracy = (prediction == y).float().mean()
    print(f"epoch: {epoch + 1}, loss: {loss.item():.4f}, accuracy: {accuracy.item():.2%}")





epoch: 1, loss: 2.4754, accuracy: 42.19%
epoch: 2, loss: 2.4398, accuracy: 43.75%
epoch: 3, loss: 2.4066, accuracy: 42.19%
epoch: 4, loss: 2.3756, accuracy: 42.19%
epoch: 5, loss: 2.3465, accuracy: 42.19%
epoch: 6, loss: 2.3193, accuracy: 42.19%
epoch: 7, loss: 2.2936, accuracy: 42.19%
epoch: 8, loss: 2.2693, accuracy: 42.19%
epoch: 9, loss: 2.2464, accuracy: 42.19%
epoch: 10, loss: 2.2249, accuracy: 45.31%
epoch: 11, loss: 2.2047, accuracy: 45.31%
epoch: 12, loss: 2.1858, accuracy: 46.88%
epoch: 13, loss: 2.1682, accuracy: 45.31%
epoch: 14, loss: 2.1519, accuracy: 45.31%
epoch: 15, loss: 2.1368, accuracy: 45.31%
epoch: 16, loss: 2.1229, accuracy: 45.31%
epoch: 17, loss: 2.1100, accuracy: 46.88%
epoch: 18, loss: 2.0982, accuracy: 46.88%
epoch: 19, loss: 2.0872, accuracy: 46.88%
epoch: 20, loss: 2.0771, accuracy: 46.88%
epoch: 21, loss: 2.0676, accuracy: 46.88%
epoch: 22, loss: 2.0587, accuracy: 46.88%
epoch: 23, loss: 2.0503, accuracy: 46.88%
epoch: 24, loss: 2.0421, accuracy: 46.88%
e

짝수/홀수를 SNN으로 학습시키고 싶었습니다.
accurancy가 50% 부근인 게 왜인가 싶어서 GPT에게 물어봤습니다.
나머지 정보를 주라는 힌트를 받았습니다.
`x = (torch.cat((evenX, oddX), dim=0)%2).float().to(device)`
짧은 생각으로 단순하게 이렇게 코드를 수정해봤습니다. 당연하게도 더 엉망인 결과가 나옵니다. 조금만 생각해보면 0,1밖에 남지 않아서 원본 데이터가 완전히 유실된다는 것을 깨달았습니다.

원본 데이터에 추가 정보를 붙이는 방법이 궁금해졌습니다.
`x = torch.cat((original_x, remainder_x), dim=1).float().to(device)`
답변 받은 예시입니다.
여기서 약간의 의문점이 들었습니다. evenX, oddX 때와는 다르게 dim=1입니다. 전에는 하나의 row에 붙이기 때문에 dim=0이고, 이번엔 데이터에 하나의 row를 추가하기 때문이라고 추측했습니다. 하지만 dim은 어떤 축을 더할 것인지를 나타냅니다. dim=0이면 shape에서 dim=0에 해당하는 것이 커집니다.

In [8]:
import torch
import torch.nn as nn
import snntorch as snn

device = torch.device("mps")

class SNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(20, 2)
        self.leaky = snn.Leaky(beta=0.9)

    def forward(self, x):
        mem = self.leaky.init_leaky()

        current = self.fc(x)
        spike, mem = self.leaky(current, mem)

        return spike, mem

model = SNN().to(device)

evenX = torch.randint(0, 10, (32, 10)) * 2
oddX = torch.randint(0, 10, (32, 10)) * 2 + 1

evenY = torch.ones(32, dtype=torch.long)
oddY = torch.zeros(32, dtype=torch.long)

x = torch.cat((evenX, oddX), dim=0)
remainderX = x % 2
x = torch.cat((x, remainderX), dim=1).float().to(device)
y = torch.cat ((evenY, oddY), dim=0).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters())

model.train()

for epoch in range(100):
    optimizer.zero_grad()

    spike, mem = model(x)
    loss = criterion(mem, y)

    loss.backward()
    optimizer.step()

    prediction = mem.argmax(dim=1)
    accuracy = (prediction == y).float().mean()
    print(f"epoch: {epoch + 1}, loss: {loss.item():.4f}, accuracy: {accuracy.item():.2%}")





epoch: 1, loss: 1.0380, accuracy: 43.75%
epoch: 2, loss: 1.0183, accuracy: 43.75%
epoch: 3, loss: 1.0027, accuracy: 45.31%
epoch: 4, loss: 0.9904, accuracy: 48.44%
epoch: 5, loss: 0.9801, accuracy: 50.00%
epoch: 6, loss: 0.9704, accuracy: 51.56%
epoch: 7, loss: 0.9606, accuracy: 53.12%
epoch: 8, loss: 0.9502, accuracy: 53.12%
epoch: 9, loss: 0.9393, accuracy: 53.12%
epoch: 10, loss: 0.9281, accuracy: 53.12%
epoch: 11, loss: 0.9166, accuracy: 53.12%
epoch: 12, loss: 0.9052, accuracy: 51.56%
epoch: 13, loss: 0.8940, accuracy: 51.56%
epoch: 14, loss: 0.8830, accuracy: 51.56%
epoch: 15, loss: 0.8724, accuracy: 51.56%
epoch: 16, loss: 0.8621, accuracy: 53.12%
epoch: 17, loss: 0.8520, accuracy: 51.56%
epoch: 18, loss: 0.8420, accuracy: 51.56%
epoch: 19, loss: 0.8320, accuracy: 51.56%
epoch: 20, loss: 0.8220, accuracy: 51.56%
epoch: 21, loss: 0.8120, accuracy: 53.12%
epoch: 22, loss: 0.8020, accuracy: 53.12%
epoch: 23, loss: 0.7921, accuracy: 53.12%
epoch: 24, loss: 0.7824, accuracy: 53.12%
e

정확도가 많이 올라갔습니다. 하지만 코드를 작성하며 떠오른 것은 짝수/홀수 판별을 SNN으로 하는 것은 그리 좋은 생각이 아닌 것 같습니다. 심지어 지금과 같은 방식은 2,4,8,...,1,1,1,...,1 과 같이 데이터의 끝에 사실상 정답을 던져주고 있다고 생각합니다.

그래도 오늘 연습으로 직접 만들어보려는 노력으로 조금은 이해도가 올랐을 거라 생각합니다.

In [10]:
for epoch in range(7000):
    optimizer.zero_grad()

    spike, mem = model(x)
    loss = criterion(mem, y)

    loss.backward()
    optimizer.step()

    if (epoch % 500 == 0):
        prediction = mem.argmax(dim=1)
        accuracy = (prediction == y).float().mean()
        print(f"epoch: {epoch}, loss: {loss.item():.4f}, accuracy: {accuracy.item():.2%}")


epoch: 0, loss: 0.0015, accuracy: 100.00%
epoch: 500, loss: 0.0011, accuracy: 100.00%
epoch: 1000, loss: 0.0008, accuracy: 100.00%
epoch: 1500, loss: 0.0006, accuracy: 100.00%
epoch: 2000, loss: 0.0005, accuracy: 100.00%
epoch: 2500, loss: 0.0004, accuracy: 100.00%
epoch: 3000, loss: 0.0003, accuracy: 100.00%
epoch: 3500, loss: 0.0002, accuracy: 100.00%
epoch: 4000, loss: 0.0002, accuracy: 100.00%
epoch: 4500, loss: 0.0001, accuracy: 100.00%
epoch: 5000, loss: 0.0001, accuracy: 100.00%
epoch: 5500, loss: 0.0001, accuracy: 100.00%
epoch: 6000, loss: 0.0001, accuracy: 100.00%
epoch: 6500, loss: 0.0000, accuracy: 100.00%
